# VK-LSVD → inter.json + content_embeddings.pkl

Аналог `notebooks/DatasetProcessing.ipynb` для VK-LSVD. На выходе:
* `../data/VK/inter.json` — интеракции в формате `{str(user_id): [item_id, ...]}`
* `../data/VK/content_embeddings.pkl` — `{'item_id': [...], 'embedding': [...]}` со встроенными VK эмбедами.

Шаги: скачивание сабсэмпла `ur0.01_ir0.01` через `hf download` (как в `LsvdDownload.ipynb`), фильтрация по `timespent > 15`, пересечение с эмбедами, **итеративный Core-5 фильтр**, ремап `user_id`/`item_id` в плотный 0-индекс, группировка по юзеру.


In [ ]:
%pip install polars==1.36.1
%pip install numpy
%pip install pyarrow
%pip install huggingface_hub

In [ ]:
import json
import os
import pickle
from collections import defaultdict

import numpy as np
import polars as pl

## Конфигурация

Пути и параметры. На A100-сервере оставляем `/home/jovyan/...`. На других машинах — переопределяем `DATASET_PATH`.

In [ ]:
SUBSAMPLE = 'ur0.01_ir0.01'
DATASET_PATH = '/home/jovyan/vkml/data/vk_lsvd/raw'
POSITIVE_EVENT_TIMESPENT = 15
CORE_K = 5

INTERACTIONS_OUTPUT_PATH = '../data/VK/inter.json'
EMBEDDINGS_OUTPUT_PATH = '../data/VK/content_embeddings.pkl'

os.makedirs(os.path.dirname(INTERACTIONS_OUTPUT_PATH), exist_ok=True)

## Скачивание (выполнять один раз)

Ячейки ниже — копия `LsvdDownload.ipynb`. Если уже скачано в `DATASET_PATH`, можно пропустить.

In [ ]:
!HF_ENDPOINT="http://huggingface.proxy" hf download deepvk/VK-LSVD --repo-type dataset --include "metadata/*" --local-dir {DATASET_PATH}

In [ ]:
!HF_ENDPOINT="http://huggingface.proxy" hf download deepvk/VK-LSVD --repo-type dataset --include "subsamples/{SUBSAMPLE}/*" --local-dir {DATASET_PATH}

## Чтение интеракций (все 25 недель)

Конкатенируем base+gap+val+test (как договорились — leave-one-out внутри TIGER делается по позиции, поэтому в `inter.json` идёт вся история).

In [ ]:
ALL_WEEKS = list(range(25))
all_files = [f'subsamples/{SUBSAMPLE}/train/week_{i:02}.parquet' for i in ALL_WEEKS]
len(all_files)

In [ ]:
def get_parquet_interactions(data_files, positive_event_timespent):
    df = pl.concat([pl.scan_parquet(f'{DATASET_PATH}/{f}') for f in data_files]).collect()
    df = df.with_row_index('original_order')
    df = df.filter(pl.col('timespent') > positive_event_timespent)
    return df

all_inter = get_parquet_interactions(all_files, POSITIVE_EVENT_TIMESPENT)
print('после timespent-фильтра:', all_inter.shape)

## Пересечение с эмбедами

Выбрасываем интеракции с айтемами, у которых нет эмбеддинга в `metadata/item_embeddings.npz`.

In [ ]:
emb_npz = np.load(f'{DATASET_PATH}/metadata/item_embeddings.npz')
emb_item_ids = emb_npz['item_id']
emb_vectors = emb_npz['embedding']
print('эмбедов всего:', emb_item_ids.shape, emb_vectors.shape)

In [ ]:
items_with_emb = pl.DataFrame({'item_id': emb_item_ids})
filtered_df = all_inter.join(items_with_emb, on='item_id', how='inner')
print('после пересечения с эмбедами:', filtered_df.shape)

## Core-5 фильтрация

Итеративно выкидываем юзеров и айтемы с <5 интеракций до сходимости (как в `DatasetProcessing.ipynb`).

In [ ]:
is_changed = True
iteration = 0
while is_changed:
    iteration += 1
    user_counts = filtered_df.group_by('user_id').agg(pl.len().alias('user_count'))
    item_counts = filtered_df.group_by('item_id').agg(pl.len().alias('item_count'))

    good_users = user_counts.filter(pl.col('user_count') >= CORE_K).select('user_id')
    good_items = item_counts.filter(pl.col('item_count') >= CORE_K).select('item_id')

    old_size = len(filtered_df)
    new_df = filtered_df.join(good_users, on='user_id', how='inner')
    new_df = new_df.join(good_items, on='item_id', how='inner')
    new_size = len(new_df)

    filtered_df = new_df
    is_changed = old_size != new_size
    print(f'iter {iteration}: {old_size} -> {new_size}')

print('финал после Core-5:', filtered_df.shape)

## Ремап user_id и item_id в 0-indexed dense

Сохраняем порядок появления — это даст компактные id.

In [ ]:
unique_users = filtered_df['user_id'].unique(maintain_order=True).to_list()
user_ids_mapping = {value: i for i, value in enumerate(unique_users)}

unique_items = filtered_df['item_id'].unique(maintain_order=True).to_list()
item_ids_mapping = {value: i for i, value in enumerate(unique_items)}

num_users = len(user_ids_mapping)
num_items = len(item_ids_mapping)
print('num_users:', num_users, 'num_items:', num_items)

In [ ]:
filtered_df = filtered_df.with_columns([
    pl.col('user_id').replace_strict(user_ids_mapping).alias('user_id'),
    pl.col('item_id').replace_strict(item_ids_mapping).alias('item_id'),
])
filtered_df.head()

## Группировка по юзеру и сериализация inter.json

Сортируем по `original_order` (это и есть таймстемп в LSVD), чтобы в листе айтемов сохранялся хронологический порядок.

In [ ]:
filtered_df = filtered_df.sort(['user_id', 'original_order'])
grouped = (
    filtered_df
    .group_by('user_id', maintain_order=True)
    .agg(pl.col('item_id'))
)
grouped.head()

In [ ]:
json_data = {}
for user_id, item_list in grouped.iter_rows():
    json_data[int(user_id)] = list(map(int, item_list))

# sanity: Core-5 после ремапа должен сохраниться
assert all(len(v) >= CORE_K for v in json_data.values()), 'Core-5 broken после ремапа'
assert max(max(v) for v in json_data.values()) == num_items - 1
assert min(min(v) for v in json_data.values()) == 0

with open(INTERACTIONS_OUTPUT_PATH, 'w') as f:
    json.dump(json_data, f, indent=2)

print(f'inter.json: {len(json_data)} юзеров, диапазон item_id [0, {num_items - 1}]')

## Сохранение content_embeddings.pkl

Переупорядочиваем `item_embeddings.npz` под новый item_id mapping. Формат — как в Amazon-пайплайне: `dict('item_id' -> list[int], 'embedding' -> list[np.ndarray(D,) float32])`.

In [ ]:
old_to_new = item_ids_mapping  # {old_item_id: new_item_id} (подмножество от npz)

# индекс эмбедов по старому item_id
old_id_to_pos = {int(old): i for i, old in enumerate(emb_item_ids)}

new_item_ids = list(range(num_items))
new_embeddings = np.zeros((num_items, emb_vectors.shape[1]), dtype=np.float32)
for old_id, new_id in old_to_new.items():
    new_embeddings[new_id] = emb_vectors[old_id_to_pos[int(old_id)]].astype(np.float32)

print('new_embeddings.shape:', new_embeddings.shape)
print('non-zero rows:', int(np.any(new_embeddings != 0, axis=1).sum()))

In [ ]:
out = {
    'item_id': new_item_ids,
    'embedding': [new_embeddings[i] for i in range(num_items)],
}
with open(EMBEDDINGS_OUTPUT_PATH, 'wb') as f:
    pickle.dump(out, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f'content_embeddings.pkl сохранён: {EMBEDDINGS_OUTPUT_PATH}')
print(f'  num_items: {len(out["item_id"])}, D: {out["embedding"][0].shape[0]}')

## Финальные sanity-чеки

In [ ]:
with open(INTERACTIONS_OUTPUT_PATH) as f:
    inter = json.load(f)
with open(EMBEDDINGS_OUTPUT_PATH, 'rb') as f:
    emb = pickle.load(f)

assert set(map(int, inter.keys())) == set(range(len(inter)))
assert emb['item_id'] == list(range(len(emb['item_id'])))
assert max(max(v) for v in inter.values()) == len(emb['item_id']) - 1
print('OK:', len(inter), 'юзеров,', len(emb['item_id']), 'айтемов,', emb['embedding'][0].shape, 'эмбед')